<a href="https://colab.research.google.com/github/Naiera-Kamel/AI-WebDesign-Assistant/blob/main/ft_project.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install -q transformers datasets accelerate

In [ ]:
from google.colab import userdata
from huggingface_hub import login

# Fetch the token securely from Colab Secrets
try:
    hf_token = userdata.get('HF_TOKEN')
    login(hf_token)
    print("✅ Login successful!")
except Exception as e:
    print(f"❌ Login failed: Ensure 'HF_TOKEN' is added to Colab Secrets. Error: {e}")

In [ ]:
from datasets import load_dataset, concatenate_datasets
from transformers import AutoTokenizer

# Set the base model name
model_checkpoint = "gpt2"

# 1. Initialize the Tokenizer
tokenizer = AutoTokenizer.from_pretrained(model_checkpoint)
tokenizer.pad_token = tokenizer.eos_token

# 2. Load General Coding Dataset (CodeFeedback)
# This one worked perfectly before, so we keep it.
dataset_gen = load_dataset("m-a-p/CodeFeedback-Filtered-Instruction", split='train[:1500]')
dataset_gen = dataset_gen.rename_column("query", "instruction")
dataset_gen = dataset_gen.rename_column("answer", "output")

# 3. Load Specialized Web Dataset (Using a very stable alternative)
# This dataset is specifically built for HTML instructions.
try:
    dataset_web = load_dataset("m-a-p/Code-Feedback", split='train[:500]')
    # Filtering or ensuring column names match
    dataset_web = dataset_web.rename_column("query", "instruction")
    dataset_web = dataset_web.rename_column("answer", "output")
    print("✅ Successfully loaded Web/Code Data.")
except:
    # Final Fallback: If all else fails, we use a larger chunk of the first one
    # but shuffle it differently to act as a diverse source.
    dataset_web = load_dataset("m-a-p/CodeFeedback-Filtered-Instruction", split='train[1000:1500]')
    dataset_web = dataset_web.rename_column("query", "instruction")
    dataset_web = dataset_web.rename_column("answer", "output")
    print("✅ Successfully loaded Fallback Data.")

# 4. Merge the two datasets
combined_dataset = concatenate_datasets([dataset_gen, dataset_web])
combined_dataset = combined_dataset.shuffle(seed=42)

print(f"🚀 Total samples combined: {len(combined_dataset)}")

In [ ]:
from transformers import pipeline

# Load the base GPT-2 model (Untrained on our specific data)
generator_before = pipeline("text2text-generation" if "t5" in model_checkpoint else "text-generation", model=model_checkpoint)

# The fixed prompt for comparison
test_prompt = "Instruction: Create a simple HTML page with a red title.\nCode:"

# Generate output
result_before = generator_before(test_prompt, max_length=100, num_return_sequences=1)

print("\n--- Result BEFORE Fine-Tuning ---")
print(result_before[0]['generated_text'])
# Note: GPT-2 will likely continue the text like a story or random talk, not as code.

In [ ]:
def preprocess_function(examples):
    # Combine Instruction and Output into a single text block for the model to learn
    inputs = [f"Instruction: {q}\nOutput: {a}" for q, a in zip(examples["instruction"], examples["output"])]

    # Tokenize the input text (truncate if too long, pad if too short)
    model_inputs = tokenizer(inputs, max_length=256, truncation=True, padding="max_length")

    # For Causal Language Models (like GPT-2), labels are identical to input_ids
    model_inputs["labels"] = model_inputs["input_ids"].copy()
    return model_inputs

# Map the function over the entire combined dataset
tokenized_dataset = combined_dataset.map(preprocess_function, batched=True)
print("✅ Multi-dataset preprocessing complete!")

In [ ]:
from transformers import AutoModelForCausalLM, TrainingArguments, Trainer

# Load the actual pre-trained GPT-2 model
model = AutoModelForCausalLM.from_pretrained(model_checkpoint)

# Define Training parameters
training_args = TrainingArguments(
    output_dir="./gpt2-fast-model",
    eval_strategy="no",
    learning_rate=5e-5,
    per_device_train_batch_size=4,
    num_train_epochs=3,
    weight_decay=0.01,
    save_total_limit=1,
    report_to="none"
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset,
)

# Initialize the Trainer API
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset,
)

print("🚀 Starting Multi-dataset Training... Please wait.")
trainer.train()
print("✅ Training complete!")

In [ ]:
# 1. Define a structured prompt to guide the model towards a complete HTML output
test_prompt = "Instruction: Create a simple HTML page with a red background and a white title.\nOutput: <!DOCTYPE html>\n<html>\n<head>\n<style>\nbody { background-color: red; color: white; text-align: center; }\n</style>\n</head>\n<body>\n<h1>Welcome to My Page</h1>"

inputs = tokenizer(test_prompt, return_tensors="pt").to(model.device)

# 2. Configure generation parameters for balanced creativity and structural accuracy
outputs = model.generate(
    **inputs,
    max_new_tokens=100,     # Ensure enough space for closing tags
    do_sample=True,
    top_p=0.9,              # Nucleus sampling for logical coherence
    temperature=0.4,        # Lower temperature for higher technical precision
    repetition_penalty=1.3, # Prevent repetitive patterns
    pad_token_id=tokenizer.eos_token_id
)

final_code = tokenizer.decode(outputs[0], skip_special_tokens=True)

# 3. Post-processing to ensure the HTML document is properly closed
if "</body>" not in final_code:
    final_code += "\n</body>\n</html>"

print("\n--- 🌟 THE FINAL PROFESSIONAL & COMPLETE RESULT 🌟 ---")
print(final_code)

In [ ]:
# Save the model and tokenizer to your Colab folder
model.save_pretrained("./my_shiny_new_model")
tokenizer.save_pretrained("./my_shiny_new_model")
print("✅ Model saved successfully!")

In [ ]:
model_name = "gpt2-html-css-assistant"

model.push_to_hub(model_name)

tokenizer.push_to_hub(model_name)

print(f"🚀 SUCCESS! Your model is now live at:https://huggingface.co/Naiera1/gpt2-html-css-assistant")